# Reading In All JSON Files

In [1]:
import pandas as pd
import json
from pathlib import Path

data_dir = Path('live-prediction-data')
json_files = list(data_dir.rglob('*.json'))

print(f"Found {len(json_files)} JSON files across all subfolders.")

# Opening and summarizing the JSON files
def load_and_summarize():
    all_data = []
    
    for file_path in json_files:
        with open(file_path, 'r') as f:
            try:
                data = json.load(f)
                if isinstance(data, dict):
                    data['source_file'] = file_path.name
                    data['folder_date'] = file_path.parent.name
                    all_data.append(data)
                else:
                    all_data.extend(data)
            except Exception as e:
                print(f"Error parsing {file_path}: {e}")

    df = pd.DataFrame(all_data)
    
    # Basic EDA Outputs
    print("\n--- Data Shape ---")
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    
    print("\n--- Column Types & Null Values ---")
    print(df.info())
    
    print("\n--- Summary Statistics (Numeric) ---")
    print(df.describe())
    
    print("\n--- Sample Data ---")
    print(df.head())
    
    return df

df = load_and_summarize()

# Optional: Check if lat/long pairs are consistent
if 'latitude' in df.columns and 'longitude' in df.columns:
    unique_locs = df.groupby(['latitude', 'longitude']).size().reset_index().rename(columns={0:'count'})
    print(f"\nUnique Lat/Long pairs found: {len(unique_locs)}")

Found 5998 JSON files across all subfolders.

--- Data Shape ---
Rows: 5998, Columns: 11

--- Column Types & Null Values ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5998 entries, 0 to 5997
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   latitude               5998 non-null   float64
 1   longitude              5998 non-null   float64
 2   generationtime_ms      5998 non-null   float64
 3   utc_offset_seconds     5998 non-null   int64  
 4   timezone               5998 non-null   object 
 5   timezone_abbreviation  5998 non-null   object 
 6   elevation              5998 non-null   float64
 7   hourly_units           5998 non-null   object 
 8   hourly                 5998 non-null   object 
 9   source_file            5998 non-null   object 
 10  folder_date            5998 non-null   object 
dtypes: float64(4), int64(1), object(6)
memory usage: 515.6+ KB
None

--- Summary Statistics

# Further JSON Analysis and Augmentation

In [2]:
# This takes the dictionary and turns keys into columns
hourly_df = pd.json_normalize(df['hourly'])

print("--- Flattened Hourly Data Columns ---")
print(hourly_df.columns)

# To see the first 5 hours of the first file:
print("\nFirst row of hourly precipitation:")
print(hourly_df['precipitation'].iloc[0])
print(len(hourly_df['precipitation'].iloc[0]))  # Should be 24 if it's hourly data for a day

--- Flattened Hourly Data Columns ---
Index(['time', 'precipitation'], dtype='object')

First row of hourly precipitation:
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.1, 0.1, 0.5, 0.5, 0.5, 0.0, 0.0, 0.0, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
168


In [3]:
latitude = df['latitude'].iloc[0]
longitude = df['longitude'].iloc[0]
print(f"\nLatitude: {latitude}, Longitude: {longitude}")


Latitude: 48.382412, Longitude: -123.56661


In [4]:
hourly_units_df = pd.json_normalize(df['hourly_units'])
print("\nHourly Units:")
print(hourly_units_df.head())


Hourly Units:
      time precipitation
0  iso8601            mm
1  iso8601            mm
2  iso8601            mm
3  iso8601            mm
4  iso8601            mm


In [5]:
import pandas as pd

# This function slices the lists inside the 'hourly' dictionary
def slice_to_24_hours(hourly_dict):
    if isinstance(hourly_dict, dict):
        # We create a new dictionary with the same keys, but only the first 24 items
        return {key: value[:24] for key, value in hourly_dict.items()}
    return hourly_dict

# Apply the slicing
df['hourly_24h'] = df['hourly'].apply(slice_to_24_hours)

# Results
first_row_len = len(df['hourly_24h'].iloc[0]['precipitation'])
print(f"New entry length: {first_row_len} hours")

# Flattening into a tidy format
rows = []
for _, row in df.iterrows():
    h = row['hourly_24h']
    for i in range(24):
        rows.append({
            'timestamp': h['time'][i],
            'latitude': row['latitude'],
            'longitude': row['longitude'],
            'precipitation_mm': h['precipitation'][i],
            'folder_date': row.get('folder_date', 'N/A')
        })

tidy_df = pd.DataFrame(rows)

print("\n--- Flattened Tidy Data (First 5 rows) ---")
print(tidy_df.head(10))

New entry length: 24 hours

--- Flattened Tidy Data (First 5 rows) ---
          timestamp   latitude  longitude  precipitation_mm folder_date
0  2026-02-27T00:00  48.382412 -123.56661               0.0  2026-02-27
1  2026-02-27T01:00  48.382412 -123.56661               0.0  2026-02-27
2  2026-02-27T02:00  48.382412 -123.56661               0.0  2026-02-27
3  2026-02-27T03:00  48.382412 -123.56661               0.0  2026-02-27
4  2026-02-27T04:00  48.382412 -123.56661               0.0  2026-02-27
5  2026-02-27T05:00  48.382412 -123.56661               0.0  2026-02-27
6  2026-02-27T06:00  48.382412 -123.56661               0.0  2026-02-27
7  2026-02-27T07:00  48.382412 -123.56661               0.0  2026-02-27
8  2026-02-27T08:00  48.382412 -123.56661               0.0  2026-02-27
9  2026-02-27T09:00  48.382412 -123.56661               0.0  2026-02-27


In [6]:
# aggregating to daily totals
daily_df = tidy_df.groupby(['folder_date', 'latitude', 'longitude'])['precipitation_mm'].sum().reset_index()

daily_df = daily_df.rename(columns={'precipitation_mm': 'daily_precip_total_mm', 'folder_date': 'date'})

print(daily_df.head())

         date   latitude   longitude  daily_precip_total_mm
0  2026-02-27  48.331406 -123.544300                    0.0
1  2026-02-27  48.382412 -123.566610                    0.0
2  2026-02-27  48.434090 -123.297620                    0.0
3  2026-02-27  48.437534 -123.424080                    0.0
4  2026-02-27  48.583110 -123.529526                    0.0


In [7]:
print(daily_df.sample(5))

            date  latitude   longitude  daily_precip_total_mm
4803  2026-03-31  50.14696 -123.121925                   0.00
5171  2026-04-02  50.75000 -122.000000                   0.40
589   2026-03-07  50.50000 -126.000000                  62.30
3791  2026-03-25  52.37500 -126.625000                  19.40
2280  2026-03-17  49.21302 -122.670770                  19.98


# Attemmpting JSON Coordinate Remediation Through Rounding

In [8]:
import numpy as np
# Shifting coordinates to 2 decimal places (e.g., 40.7128 -> 40.71)
daily_df['latitude'] = np.floor(daily_df['latitude'] * 100) / 100
daily_df['longitude'] = np.floor(daily_df['longitude'] * 100) / 100
print(daily_df.head())

         date  latitude  longitude  daily_precip_total_mm
0  2026-02-27     48.33    -123.55                    0.0
1  2026-02-27     48.38    -123.57                    0.0
2  2026-02-27     48.43    -123.30                    0.0
3  2026-02-27     48.43    -123.43                    0.0
4  2026-02-27     48.58    -123.53                    0.0


In [9]:
mapping_df = pd.read_csv('Hydro_to_Climate_Mapping.csv')

# Normalizes coordinates to 2 decimal places 
def norm_coord(df, mapping=False):
    if not mapping:
        return df.assign(
            lat_norm = (df['latitude'] * 100).astype(int) / 100,
            lon_norm = (df['longitude'] * 100).astype(int) / 100
        )
    else:
        return df.assign(
            lat_norm = (df['Climate Lat'] * 100).astype(int) / 100,
            lon_norm = (df['Climate Long'] * 100).astype(int) / 100
        )

weather_norm = norm_coord(daily_df)
mapping_norm = norm_coord(mapping_df, mapping=True)

# Create sets of unique coordinate pairs
weather_set = set(zip(weather_norm['lat_norm'], weather_norm['lon_norm']))
mapping_set = set(zip(mapping_norm['lat_norm'], mapping_norm['lon_norm']))

# Compare
matches = weather_set.intersection(mapping_set)
missing_in_weather = mapping_set - weather_set

print(f"Total Unique Mapping Stations: {len(mapping_set)}")
print(f"Matches Found: {len(matches)}")
print(f"Stations with NO matching weather data: {len(missing_in_weather)}")

Total Unique Mapping Stations: 100
Matches Found: 8
Stations with NO matching weather data: 92


In [10]:
# Perform a outer join to see the full picture
diagnostic_df = pd.merge(
    mapping_norm[['Climate ID', 'lat_norm', 'lon_norm']], 
    weather_norm[['lat_norm', 'lon_norm']].drop_duplicates(), 
    on=['lat_norm', 'lon_norm'], 
    how='outer', 
    indicator=True
)

# Analyzing the efficacy of rounding to 2 decimal places for matching coordinates
# 'both' = Match
# 'left_only' = Station exists, but no weather JSON found
# 'right_only' = Weather data exists for a coordinate not in your station list
print(diagnostic_df['_merge'].value_counts())

_merge
left_only     208
right_only    168
both           29
Name: count, dtype: int64


# Remediating JSON Coordinates Through KDTree

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import KDTree

# Prepare the Mapping Data (The "Truth")
mapping_coords = mapping_df[['Climate Lat', 'Climate Long']].values
climate_ids = mapping_df['Climate ID'].values

# Preparing the Weather Data (The "Predictions")
unique_weather = daily_df[['latitude', 'longitude']].drop_duplicates()
weather_coords = unique_weather[['latitude', 'longitude']].values

# Build the KDTree using the Weather Grid
# We search the tree to find which Weather Point is closest to each Station
tree = KDTree(weather_coords)

# For every station in the mapping file, find the index of the nearest weather point
d, i = tree.query(mapping_coords)

# Create a Lookup Table
# This maps Station (Climate ID) to the specific Weather Coordinate it should use
lookup = pd.DataFrame({
    'Climate ID': climate_ids,
    'matched_lat': weather_coords[i, 0],
    'matched_lon': weather_coords[i, 1],
    'distance_deg': d
})

# Final Join
final_df = pd.merge(
    daily_df, 
    lookup, 
    left_on=['latitude', 'longitude'], 
    right_on=['matched_lat', 'matched_lon']
)

In [12]:
# Quick summary of distance quality
print(lookup['distance_deg'].describe())

# How many stations are "At Risk" 
at_risk = lookup[lookup['distance_deg'] > 0.15]
print(f"Stations with high spatial drift: {len(at_risk)}")
# How many Climate IDs are sharing the exact same weather data
collisions = lookup.groupby(['matched_lat', 'matched_lon'])['Climate ID'].count()
print("Top 5 Weather Points with most assigned stations:")
print(collisions.sort_values(ascending=False).head())

count    237.000000
mean       0.028084
std        0.030772
min        0.000000
25%        0.010000
50%        0.020000
75%        0.036056
max        0.167631
Name: distance_deg, dtype: float64
Stations with high spatial drift: 4
Top 5 Weather Points with most assigned stations:
matched_lat  matched_lon
49.70        -125.02        8
49.95        -125.27        7
49.96        -119.39        7
49.38        -123.09        7
49.28        -122.88        6
Name: Climate ID, dtype: int64


In [13]:
print(final_df.head())
print(final_df['date'].unique())

         date  latitude  longitude  daily_precip_total_mm Climate ID  \
0  2026-02-27     48.78    -125.18                   0.35    1031316   
1  2026-02-27     48.78    -125.18                   0.35    1031316   
2  2026-02-27     48.81    -124.06                   0.00    1012055   
3  2026-02-27     48.81    -124.06                   0.00    1012055   
4  2026-02-27     48.81    -124.06                   0.00    1012055   

   matched_lat  matched_lon  distance_deg  
0        48.78      -125.18      0.041231  
1        48.78      -125.18      0.041231  
2        48.81      -124.06      0.022361  
3        48.81      -124.06      0.022361  
4        48.81      -124.06      0.022361  
['2026-02-27' '2026-03-05' '2026-03-06' '2026-03-07' '2026-03-08'
 '2026-03-09' '2026-03-10' '2026-03-11' '2026-03-12' '2026-03-13'
 '2026-03-14' '2026-03-15' '2026-03-16' '2026-03-17' '2026-03-18'
 '2026-03-19' '2026-03-20' '2026-03-21' '2026-03-22' '2026-03-23'
 '2026-03-24' '2026-03-25' '2026-03-26'

In [14]:
df_weather_ids = pd.merge(final_df, mapping_df[['Climate ID', 'Hydro ID']], on='Climate ID', how='left')
print(df_weather_ids.head())

         date  latitude  longitude  daily_precip_total_mm Climate ID  \
0  2026-02-27     48.78    -125.18                   0.35    1031316   
1  2026-02-27     48.78    -125.18                   0.35    1031316   
2  2026-02-27     48.78    -125.18                   0.35    1031316   
3  2026-02-27     48.78    -125.18                   0.35    1031316   
4  2026-02-27     48.81    -124.06                   0.00    1012055   

   matched_lat  matched_lon  distance_deg Hydro ID  
0        48.78      -125.18      0.041231  08HB048  
1        48.78      -125.18      0.041231  08HB014  
2        48.78      -125.18      0.041231  08HB048  
3        48.78      -125.18      0.041231  08HB014  
4        48.81      -124.06      0.022361  08HA002  


# Acquiring Reservoir Levels of the last 30 days

In [15]:
live_hydro = pd.read_csv('BC_daily_hydrometric.csv') #https://dd.weather.gc.ca/today/hydrometric/csv/BC/daily/BC_daily_hydrometric.csv
live_hydro.columns = live_hydro.columns.str.strip()
live_hydro = live_hydro[["ID", 'Date', "Water Level / Niveau d'eau (m)"]]
live_hydro['Date'] = pd.to_datetime(live_hydro['Date'], errors='coerce', utc=True)
live_hydro['Date'] = live_hydro['Date'].dt.date

daily_avg = live_hydro.groupby(['ID', 'Date'])["Water Level / Niveau d'eau (m)"].agg(['mean'])
daily_avg['mean_shifted'] = daily_avg['mean'].clip(lower=0) + 1 # constant 1
daily_avg['log_mean'] = np.log(daily_avg['mean_shifted'])
daily_avg = daily_avg.reset_index()
daily_avg = daily_avg.rename(columns={"log_mean": "Water Level"})
daily_avg = daily_avg[['ID', 'Date', 'Water Level']]
daily_avg

,ID,Date,Water Level
0,07EA004,2026-03-06,1.182827
1,07EA004,2026-03-07,1.182326
2,07EA004,2026-03-08,1.179924
3,07EA004,2026-03-09,1.171359
4,07EA004,2026-03-10,1.162344
...,...,...,...
13622,10CD005,2026-04-01,0.665451
13623,10CD005,2026-04-02,0.666197
13624,10CD005,2026-04-03,0.666830
13625,10CD005,2026-04-04,0.667010


# Joining Reservoir Levels and Precipitation Forecasts

In [16]:
print(daily_avg.dtypes)
print('**'*20)
print(df_weather_ids.dtypes)

ID              object
Date            object
Water Level    float64
dtype: object
****************************************
date                      object
latitude                 float64
longitude                float64
daily_precip_total_mm    float64
Climate ID                object
matched_lat              float64
matched_lon              float64
distance_deg             float64
Hydro ID                  object
dtype: object


In [17]:
# Convert IDs to Strings and strip any hidden spaces
df_weather_ids['Hydro ID'] = df_weather_ids['Hydro ID'].astype(str).str.strip()
daily_avg['ID'] = daily_avg['ID'].astype(str).str.strip()

# Convert Dates to actual Datetime objects
df_weather_ids['date'] = pd.to_datetime(df_weather_ids['date'])
daily_avg['Date'] = pd.to_datetime(daily_avg['Date'])

# Strip time components (Force both to Midnight)
df_weather_ids['date'] = df_weather_ids['date'].dt.normalize()
daily_avg['Date'] = daily_avg['Date'].dt.normalize()

In [18]:
df_final = pd.merge(
    df_weather_ids, 
    daily_avg[['ID', 'Date', 'Water Level']],  # Select only the keys and the target column
    left_on=['Hydro ID', 'date'],        # Keys in df1
    right_on=['ID', 'Date'],             # Keys in df2
    how='left'                           # Keep all records from df1
)

# Optional: Remove the redundant 'ID' and 'Date' columns from the right-hand side
df_final = df_final.drop(columns=['ID', 'Date'])
print(df_final.head())

        date  latitude  longitude  daily_precip_total_mm Climate ID  \
0 2026-02-27     48.78    -125.18                   0.35    1031316   
1 2026-02-27     48.78    -125.18                   0.35    1031316   
2 2026-02-27     48.78    -125.18                   0.35    1031316   
3 2026-02-27     48.78    -125.18                   0.35    1031316   
4 2026-02-27     48.81    -124.06                   0.00    1012055   

   matched_lat  matched_lon  distance_deg Hydro ID  Water Level  
0        48.78      -125.18      0.041231  08HB048          NaN  
1        48.78      -125.18      0.041231  08HB014          NaN  
2        48.78      -125.18      0.041231  08HB048          NaN  
3        48.78      -125.18      0.041231  08HB014          NaN  
4        48.81      -124.06      0.022361  08HA002          NaN  


In [19]:
df_final.drop(columns=['latitude', 'longitude', 'distance_deg', 'Hydro ID'], inplace=True)
df_final.rename(columns={'matched_lat': 'latitude', 'matched_lon': 'longitude'}, inplace=True)
print(df_final.head())

        date  daily_precip_total_mm Climate ID  latitude  longitude  \
0 2026-02-27                   0.35    1031316     48.78    -125.18   
1 2026-02-27                   0.35    1031316     48.78    -125.18   
2 2026-02-27                   0.35    1031316     48.78    -125.18   
3 2026-02-27                   0.35    1031316     48.78    -125.18   
4 2026-02-27                   0.00    1012055     48.81    -124.06   

   Water Level  
0          NaN  
1          NaN  
2          NaN  
3          NaN  
4          NaN  


In [20]:
print(f"Unique Water Level entries: {df_final['Water Level'].nunique()}")
print("\nRandom sample of Water Level entries:")
print(df_final['Water Level'].sample(5))

Unique Water Level entries: 6847

Random sample of Water Level entries:
18564    1.543919
23797    0.664128
1483     1.088037
1122          NaN
5757     0.471875
Name: Water Level, dtype: float64


In [21]:
nan_count = df_final['Water Level'].isna().sum()
print(f"Number of NaN entries in Water Level column: {nan_count}")
print(len(df_final))

Number of NaN entries in Water Level column: 2194
25809


In [22]:
df_final = df_final.dropna(subset=['Water Level'])
nan_count = df_final['Water Level'].isna().sum()
print(f"Number of NaN entries in Water Level column: {nan_count}")
print(len(df_final))

Number of NaN entries in Water Level column: 0
23615


In [23]:
df_final.to_csv('modern_forecast_hydro_data.csv', index=False)
print(df_final.head())

           date  daily_precip_total_mm Climate ID  latitude  longitude  \
1419 2026-03-06                  22.48    1031316     48.78    -125.18   
1420 2026-03-06                  22.48    1031316     48.78    -125.18   
1421 2026-03-06                  22.48    1031316     48.78    -125.18   
1422 2026-03-06                  22.48    1031316     48.78    -125.18   
1423 2026-03-06                  13.78    1012055     48.81    -124.06   

      Water Level  
1419     0.194491  
1420     1.038958  
1421     0.194491  
1422     1.038958  
1423     0.820644  
